In [8]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_febros_fractions_df,
    wade_febros_scaler,
    wade_febros_pca,
    wade_febros_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-02-14 00:00:00",
    end_date="2023-02-20 00:00:00",
    endmember_ids=["RI23-5006", "RI23-5018", "RI23-5000", "RI23-5005"],
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5006", "RI23-5018", "RI23-5000", "RI23-5005"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-02-14 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-02-20 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=wade_febros_endmembers_df,
    em_raw=em_raw_subset,
    tracers=wade_tracers,
    analytical_sd=analytical_sd,
    confidence_level = 0.95 
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    wade_febros_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1009 2023-02-15 15:00:00
1  RI23-1010 2023-02-15 19:00:00
2  RI23-1011 2023-02-15 23:00:00
3  RI23-1025 2023-02-15 12:00:00
4  RI23-1012 2023-02-16 03:00:00


,Sample ID,Datetime,Site,Groundwater,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Groundwater_Uncertainty_95sig,Snowmelt lysimeter_Uncertainty_95sig,Soil water lysimeter_Uncertainty_95sig
0,RI23-1009,2023-02-15 15:00:00,Wade,0.577292,-6.342797e-14,0.422708,1.0,1.677766,2.841695e-06,1.677768
1,RI23-1010,2023-02-15 19:00:00,Wade,0.531795,3.879134e-13,0.468205,1.0,1.810438,3.595174e-09,1.810438
2,RI23-1011,2023-02-15 23:00:00,Wade,0.495768,-2.555990e-14,0.504232,1.0,2.029665,1.116050e-08,2.029665
3,RI23-1025,2023-02-15 12:00:00,Wade,0.524279,-4.660879e-13,0.475721,1.0,1.660788,7.290277e-06,1.660781
4,RI23-1012,2023-02-16 03:00:00,Wade,0.326576,-5.429684e-16,0.673424,1.0,2.474155,3.974607e-08,2.474155
5,RI23-1014,2023-02-16 11:00:00,Wade,0.229147,1.949860e-14,0.770853,1.0,2.715008,3.360215e-08,2.715008
6,RI23-1028,2023-02-16 14:00:00,Wade,0.231564,9.723918e-14,0.768436,1.0,2.648530,4.448664e-08,2.648530
7,RI23-1029,2023-02-16 20:00:00,Wade,0.292599,-1.250834e-15,0.707401,1.0,2.639642,3.584502e-08,2.639642
8,RI23-1030,2023-02-17 02:00:00,Wade,0.258380,-6.175271e-14,0.741620,1.0,2.512898,7.644462e-09,2.512898
9,RI23-1031,2023-02-17 08:00:00,Wade,0.323434,9.032271e-15,0.676566,1.0,2.413596,1.382958e-08,2.413596


In [6]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_martherm_fractions_df,
    wade_martherm_scaler,
    wade_martherm_pca,
    wade_martherm_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-03-21 00:00:00",
    end_date="2023-03-26 00:00:00",
    endmember_ids=[
                   "RI23-5018", "RI23-5006", # Homeowner well groundwater March 2023
                   "RI23-1034", # Pre-event baseflow, labeled as GW in Wade index
                   "RI23-5005", # Snowmelt lysimeter 02/15/2023
                   "RI23-1063", # Snowmelt lysimeter 3/28
                   "RI23-5011", # Soil water lysimeter wet 4/12
                   ], 
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5006", "RI23-5018", "RI23-1034", "RI23-5005", "RI23-1063", "RI23-5011"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-21 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-03-26 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=wade_martherm_endmembers_df,
    em_raw=em_raw_subset,
    tracers=wade_tracers,
    analytical_sd=analytical_sd,
    confidence_level = 0.95
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    wade_martherm_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1039 2023-03-22 18:00:00
1  RI23-1040 2023-03-23 00:00:00
2  RI23-1041 2023-03-23 06:00:00
3  RI23-1058 2023-03-24 06:00:00
4  RI23-1059 2023-03-24 12:00:00


,Sample ID,Datetime,Site,Groundwater,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Groundwater_Uncertainty_95sig,Snowmelt lysimeter_Uncertainty_95sig,Soil water lysimeter_Uncertainty_95sig
0,RI23-1039,2023-03-22 18:00:00,Wade,0.654052,0.132299,0.213649,1.0,0.652949,1.078822,1.442998
1,RI23-1040,2023-03-23 00:00:00,Wade,0.481331,0.111952,0.406718,1.0,0.686821,1.178742,1.615589
2,RI23-1041,2023-03-23 06:00:00,Wade,0.584524,0.218052,0.197424,1.0,0.625030,1.139904,1.469779
3,RI23-1058,2023-03-24 06:00:00,Wade,NaN,NaN,NaN,0.0,0.501912,1.833777,2.162434
4,RI23-1059,2023-03-24 12:00:00,Wade,NaN,NaN,NaN,0.0,0.610952,1.513478,1.836548


In [7]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_fmelt_fractions_df,
    wade_fmelt_scaler,
    wade_fmelt_pca,
    wade_fmelt_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-03-30 00:00:00",
    end_date="2023-04-15 00:00:00",
    endmember_ids=[
                   "RI23-5018", "RI23-5006", # Homeowner well groundwater March 2023
                   "RI23-5005", # Snowmelt lysimeter 02/15/2023
                   "RI23-1063", # Snowmelt lysimeter 3/28
                   "RI23-5011", # Soil water lysimeter wet 4/12
                   ],  
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5006", "RI23-5018", "RI23-5005", "RI23-1063", "RI23-5011"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-30 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-04-15 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=wade_fmelt_endmembers_df,
    em_raw=em_raw_subset,
    tracers=wade_tracers,
    analytical_sd=analytical_sd,
    confidence_level = 0.95
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    wade_fmelt_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1064 2023-03-31 08:00:00
1  RI23-1065 2023-03-31 14:00:00
2  RI23-1066 2023-03-31 20:00:00
3  RI23-1067 2023-04-01 02:00:00
4  RI23-1068 2023-04-01 08:00:00


,Sample ID,Datetime,Site,Groundwater,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Groundwater_Uncertainty_95sig,Snowmelt lysimeter_Uncertainty_95sig,Soil water lysimeter_Uncertainty_95sig
0,RI23-1064,2023-03-31 08:00:00,Wade,5.353060e-01,0.195701,0.268993,1.0,0.940137,1.451149,2.002771
1,RI23-1065,2023-03-31 14:00:00,Wade,3.678574e-01,0.181642,0.450501,1.0,0.711988,1.758261,2.214279
2,RI23-1066,2023-03-31 20:00:00,Wade,4.954517e-01,0.216167,0.288382,1.0,0.741521,1.531410,1.950834
3,RI23-1067,2023-04-01 02:00:00,Wade,5.174244e-01,0.196880,0.285696,1.0,0.787805,1.504259,1.973577
4,RI23-1068,2023-04-01 08:00:00,Wade,5.124051e-01,0.189385,0.298210,1.0,0.729602,1.477473,1.854215
5,RI23-1070,2023-04-01 20:00:00,Wade,2.158799e-01,0.280518,0.503602,1.0,0.538802,1.871080,2.120093
6,RI23-1093,2023-04-10 06:00:00,Wade,1.491495e-01,0.263434,0.587416,1.0,0.519234,1.918491,2.208293
7,RI23-1094,2023-04-10 12:00:00,Wade,7.035901e-02,0.291464,0.638177,1.0,0.487774,2.006152,2.277218
8,RI23-1095,2023-04-10 18:00:00,Wade,1.899379e-02,0.318042,0.662965,1.0,0.456849,2.105752,2.390068
9,RI23-1097,2023-04-11 06:00:00,Wade,4.392395e-15,0.416496,0.583504,1.0,0.433580,2.184272,2.475230
